# DWBI Week 3 — Fact & Dimension Tables
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's flat `Shipments_Unified` (34 rows, 578,650 kg) becomes **one fact table + five dimensions** — and the totals do not move by a single kilogram.

## T4 — Build DimCity

The editorial act of dimensional modelling: 23 raw location strings → **14 conformed cities**.

| Raw strings → City | Province | Region | Port? | Facility |
|---|---|---|---|---|
| Karachi Port, Port Qasim, Karachi Cantt, Pipri Marshalling Yard, Karachi → **Karachi** (5 aliases) | Sindh | South | Yes | Port |
| Gwadar Port → **Gwadar** | **Balochistan** ⚠️ | West | Yes | Port |
| Lahore, Lahore Dry Port → **Lahore** | Punjab | Central | No | Dry Port |
| Islamabad, Islamabad Dry Port → **Islamabad** | **Islamabad ICT** ⚠️ | Central | No | Dry Port |
| Peshawar, Peshawar Cantt → **Peshawar** | Khyber Pakhtunkhwa | North | No | City |
| Faisalabad, Faisalabad Dry Port → **Faisalabad** | Punjab | Central | No | Dry Port |
| Sialkot, Sialkot Dry Port → **Sialkot** | Punjab | Central | No | Dry Port |
| Multan / Quetta / Rawalpindi / Hyderabad / Sukkur / Gujranwala / Abbottabad | (as expected) | | No | City |

Plus the **`-1` Unknown** row — every dimension gets one. Surrogate keys 1–14 minted by hand.

**Judgement Q1** — merging Pipri (40 km out) into Karachi quietly decided that "Karachi" for network
planning *includes* the yard: a capacity decision for Karachi unknowingly commits Pipri's throughput,
and no report can ever show the yard alone again.

**Judgement Q2** — Islamabad is ICT, not Punjab. Filed under Punjab, Punjab/Road reads 50,200 kg not
44,100 and the ICT vanishes. Left blank, the 6,100 kg still reconciles in the grand total but disappears
from every province slice — a quieter, worse failure.

**Judgement Q3** — a new spelling next month, in order: (1) the lookup fails → `IFERROR` parks the row
on **-1 Unknown** (visible, countable, nothing vanishes); (2) the conformer maps the spelling and defends
the city; (3) refresh — the fact row resolves to a real key. Never hand-edit the fact; fix the map, refresh.

In [1]:
import pandas as pd
c = pd.read_csv("data/DimCity.csv")
print(f"DimCity: {len(c)} rows ({len(c)-1} cities + Unknown)")
print("Karachi aliases:", c.loc[c.CityName=='Karachi','SourceAliases'].iloc[0])
print("\nBy province:")
print(c[c.CityKey!=-1].groupby("Province").CityName.count().to_string())

DimCity: 15 rows (14 cities + Unknown)
Karachi aliases: Karachi Port | Port Qasim | Karachi Cantt | Pipri Marshalling Yard | Karachi

By province:
Province
Balochistan           2
Islamabad ICT         1
Khyber Pakhtunkhwa    2
Punjab                6
Sindh                 3
